# Modelo de Probabilidad of Default 

### Importamos librerias 

In [20]:
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import  RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score


from scipy import stats

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


### Carga de variables

In [21]:
X_train = pl.read_parquet('../data/processed/X_train.parquet')
X_test = pl.read_parquet('../data/processed/X_test.parquet')

y_train = pl.read_parquet('../data/processed/y_train.parquet')
y_test = pl.read_parquet('../data/processed/y_test.parquet')


**TAXONOMIA DE VARIABLES**

In [22]:
TARGET = "y"

LEAKAGE_VARS = ['grade']

NUM = [ col for col, dtype in X_train.schema.items() if dtype.is_numeric()]

CAT = [col for col, dtype in X_train.schema.items() if dtype == pl.String]

VARS = NUM + CAT

print(f"Variables de originación: {len(VARS)}  (numéricas: {len(NUM)}, categóricas: {len(CAT)})")


Variables de originación: 47  (numéricas: 37, categóricas: 10)


In [23]:
print(y_train["y"].value_counts(normalize=True))

shape: (2, 2)
┌─────┬────────────┐
│ y   ┆ proportion │
│ --- ┆ ---        │
│ i64 ┆ f64        │
╞═════╪════════════╡
│ 1   ┆ 0.13073    │
│ 0   ┆ 0.86927    │
└─────┴────────────┘


In [24]:
X_train = X_train.to_pandas()
X_test = X_test.to_pandas()

y_train = y_train['y'].to_numpy()
y_test = y_test['y'].to_numpy()

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),NUM),
        ("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=True),CAT)
    ]
)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

In [26]:
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [27]:
logistic_pipeline.fit(X_train,y_train)

y_prob = logistic_pipeline.predict_proba(X_test)[:,1]
y_pred = logistic_pipeline.predict(X_test)
feature_names = logistic_pipeline.named_steps["preprocessor"].get_feature_names_out()

coefficients = logistic_pipeline.named_steps["model"].coef_[0]

# Coeficientes del modelo
coef_df = pd.DataFrame({
    "Variable": feature_names,
    "Coeficiente": coefficients
}).sort_values(
    by="Coeficiente",
    key=abs,
    ascending=False
)

coef_df.head(20)

,Variable,Coeficiente
90,cat__loan_purpose_wedding,-0.797712
145,cat__disbursement_method_DirectPay,-0.736905
69,cat__emp_length_n/a,0.701405
56,cat__emp_title_UNKNOWN,-0.540525
144,cat__disbursement_method_Cash,0.455712
112,cat__addr_state_ME,-0.437988
26,num__issue_date_year,-0.429843
137,cat__addr_state_VT,-0.407310
1,num__interest_rate,0.382460
140,cat__addr_state_WV,-0.358319


In [28]:
marginal_effects = []

for i, var in enumerate(feature_names):
    me = np.mean(coefficients[i] * y_prob * (1 - y_prob))
    marginal_effects.append(me)

# Crear DataFrame para visualizar
df_marginal = pd.DataFrame({
    'Variable': feature_names,
    'Coeficiente': coefficients,
    'Efecto_Marginal_Promedio': marginal_effects
})

df_marginal.head(20)

,Variable,Coeficiente,Efecto_Marginal_Promedio
0,num__funded_amnt,-0.124362,-0.025832
1,num__interest_rate,0.382460,0.079443
2,num__monthly_payment,0.261421,0.054301
3,num__annual_income,0.006300,0.001309
4,num__dept_paym_income_ratio,0.143545,0.029817
5,num__num_30+_delinq_in_2yrs,0.035474,0.007368
6,num__num_inq_in_6mths,0.067571,0.014036
7,num__mths_since_last_delinq,-0.016160,-0.003357
8,num__num_open_credit_lines,0.067711,0.014065
9,num__num_derogatory_pub_rec,0.074031,0.015377


In [29]:
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    brier_score_loss
)

roc_auc = roc_auc_score(y_test, y_prob)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_prob)

print(f"ROC-AUC    : {roc_auc:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"F1         : {f1:.4f}")
print(f"Brier Score: {brier:.4f}")

ROC-AUC    : 0.7277
Recall     : 0.6736
Precision  : 0.2275
F1         : 0.3401
Brier Score: 0.2133
